In [71]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import TextSplitter,RecursiveCharacterTextSplitter
from mlflow import sentence_transformers
from sentence_transformers import SentenceTransformer
import chromadb
from streamlit import context

In [72]:
data = TextLoader("retention_knowledge.txt")
data

In [3]:
data_loader = data.load()
data_loader[0].page_content

'Rule 1:\nCustomers who have not ordered recently\nmay need a re-engagement campaign.\n\nRule 2:\nCustomers with declining order frequency\nmay benefit from personalized offers.\n\nRule 3:\nHigh-value customers at risk of churn\nmay need priority support.'

In [41]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_text(data_loader[0].page_content)
chunks

['Rule 1:\nCustomers who have not ordered recently\nmay need a re-engagement campaign.',
 'Rule 2:\nCustomers with declining order frequency\nmay benefit from personalized offers.',
 'Rule 3:\nHigh-value customers at risk of churn\nmay need priority support.']

In [42]:
sentence_transformers = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_transformers.encode(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19869.08it/s]


In [43]:
chromadb_client = chromadb.PersistentClient(path='/chromadb_client')
collection = chromadb_client.get_or_create_collection(name = "chunk_rag_collection")
ids = [f"id_{i}" for i in range(len(chunks))]
collection.add(
    ids = ids,
    embeddings = embeddings,
    documents=chunks,
)

In [53]:
customer_details = """
Churn probability: 0.82
Days since last order: 95
Total orders: 3
Customer tenure: 180 days
"""

query = f"""
Find retention guidance relevant to this customer:
{customer_details}
"""
query_encoded = sentence_transformers.encode(query)

In [63]:
results = collection.query(query_embeddings=[query_encoded.tolist()],n_results = 2)
results

{'ids': [['id_28', 'id_12']],
 'embeddings': None,
 'documents': [['of churn', 'Customers']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.9977377653121948, 1.3040632009506226]]}

In [64]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

In [65]:
checkpoint = 'google/flan-t5-base'

In [66]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 590.66it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [67]:
context = " ".join(results['documents'][0])
prompt = f"""
    Answer the question using the following context.

Context:
{context}

Question:
{query}

Answer:
"""

In [68]:
input = tokenizer(prompt,return_tensors = "pt")
input

{'input_ids': tensor([[11801,     8,   822,   338,     8,   826,  2625,     5,  1193,  6327,
            10,    13,     3,   524,   450,    29, 16423, 11860,    10,  2588,
         14344,  4864,  2193,    12,    48,   884,    10, 16636,    52,    29,
         15834,    10,  4097,  4613, 13992,   437,   336,   455,    10, 11923,
          9273,  5022,    10,   220,  7327, 20752,    10,  8003,   477, 11801,
            10,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]])}

In [69]:
outputs = model.generate(input_ids=input['input_ids'])

In [70]:
ans = tokenizer.decode(outputs[0],skip_special_tokens=True)
ans

'Customer tenure: 180 days'